# 04 — Tableau Export
**Project:** OULAD Learning Analytics — Portfolio Project

This notebook exports clean, Tableau-ready CSV files used to build the final dashboard views in Tableau Public.

## Exports
| File | Tableau View |
|------|-------------|
| `tableau_funnel.csv` | Conversion Funnel Bar Chart |
| `tableau_cohort_retention.csv` | Cohort Retention Heatmap |
| `tableau_geo.csv` | Geographic Distribution (UK Regions) |
| `tableau_engagement_vs_result.csv` | Engagement vs Outcome Scatter |
| `tableau_weekly_activity.csv` | Weekly Activity Timeline |

In [1]:
import pandas as pd
import numpy as np
import os

PROC    = '../data/processed/'
RAW     = '../data/raw/'
TABLEAU = '../outputs/tableau/'
os.makedirs(TABLEAU, exist_ok=True)

master      = pd.read_csv(PROC + 'master_students.csv')
student_vle = pd.read_csv(RAW + 'studentVle.csv')
vle         = pd.read_csv(RAW + 'vle.csv')  # activity type metadata

print(f'Master: {master.shape}')
print(f'Student VLE: {student_vle.shape}')

Master: (32593, 30)
Student VLE: (10655280, 6)


---
## Export 1: Funnel Data

In [2]:
total         = len(master)
n_active      = (master['total_clicks'] > 0).sum()
n_submitted   = (master['submission_count'] > 0).sum()
n_passed      = master['completed'].sum()
n_distinction = (master['final_result'] == 'Distinction').sum()

funnel_df = pd.DataFrame({
    'stage_order': [1, 2, 3, 4, 5],
    'stage_name':  ['Registered', 'Active (Any Click)', 'Submitted Assessment', 'Passed', 'Distinction'],
    'students':    [total, n_active, n_submitted, n_passed, n_distinction]
})
funnel_df['pct_of_registered'] = (funnel_df['students'] / total * 100).round(1)
funnel_df['drop_off_pct'] = (100 - funnel_df['pct_of_registered']).round(1)

funnel_df.to_csv(TABLEAU + 'tableau_funnel.csv', index=False)
print('Saved tableau_funnel.csv')
funnel_df

Saved tableau_funnel.csv


,stage_order,stage_name,students,pct_of_registered,drop_off_pct
0,1,Registered,32593,100.0,0.0
1,2,Active (Any Click),29228,89.7,10.3
2,3,Submitted Assessment,25820,79.2,20.8
3,4,Passed,15385,47.2,52.8
4,5,Distinction,3024,9.3,90.7


---
## Export 2: Cohort Retention (Long Format)

This export preserves the published workbook's Week 0 denominator. Notebook 02
uses peak weekly activity in weeks 0–2 and caps the retention index at 100%.
The comparison below quantifies the differences for the workbook owner.


In [3]:
sv = student_vle.copy()
sv['week'] = (sv['date'] // 7).astype(int)
sv = sv[sv['week'].between(0, 25)]

sv = sv.merge(
    master[['id_student','code_module','code_presentation']].drop_duplicates(),
    on=['id_student','code_module','code_presentation'], how='inner'
)

weekly_active = (
    sv.groupby(['code_presentation', 'week'])['id_student']
    .nunique()
    .reset_index(name='active_students')
)

cohort_sizes = (
    weekly_active[weekly_active['week'] == 0]
    .set_index('code_presentation')['active_students']
)

weekly_active['cohort_size']    = weekly_active['code_presentation'].map(cohort_sizes)
weekly_active['retention_rate'] = (weekly_active['active_students'] / weekly_active['cohort_size'] * 100).round(1)
weekly_active.rename(columns={'code_presentation': 'cohort'}, inplace=True)

weekly_active.to_csv(TABLEAU + 'tableau_cohort_retention.csv', index=False)
print('Saved tableau_cohort_retention.csv')
weekly_active.head(10)

Saved tableau_cohort_retention.csv


,cohort,week,active_students,cohort_size,retention_rate
0,2013B,0,3615,3615,100.0
1,2013B,1,3500,3615,96.8
2,2013B,2,3740,3615,103.5
3,2013B,3,3522,3615,97.4
4,2013B,4,3296,3615,91.2
5,2013B,5,3187,3615,88.2
6,2013B,6,3380,3615,93.5
7,2013B,7,3149,3615,87.1
8,2013B,8,2748,3615,76.0
9,2013B,9,2740,3615,75.8


In [4]:
# Compare the legacy export with the method used in notebook 02.
current_counts = (
    sv.loc[sv['sum_click'].gt(0)].groupby(['code_presentation', 'week'])['id_student']
    .nunique().rename('active_students').reset_index()
)
early_peak = (
    current_counts.loc[current_counts['week'].le(2)]
    .groupby('code_presentation')['active_students'].max()
)
current_counts['peak_based_pct'] = (
    100 * current_counts['active_students'] / current_counts['code_presentation'].map(early_peak)
).clip(upper=100)
comparison = weekly_active[['cohort', 'week', 'retention_rate']].merge(
    current_counts.rename(columns={'code_presentation': 'cohort'})[['cohort', 'week', 'peak_based_pct']],
    on=['cohort', 'week'], validate='one_to_one',
).rename(columns={'retention_rate': 'tableau_pct'})

display(comparison.query("cohort == '2013B' and week <= 2").round(1))
above_100 = comparison['tableau_pct'].gt(100).sum()
print(f'Legacy export cells above 100%: {above_100} of {len(comparison)}.')


,cohort,week,tableau_pct,peak_based_pct
0,2013B,0,100.0,96.7
1,2013B,1,96.8,93.6
2,2013B,2,103.5,100.0


Legacy export cells above 100%: 3 of 104.


---
## Export 3: Geographic Distribution (UK Regions)

In [5]:
# OULAD has 'region' field in studentInfo
geo_df = (
    master
    .groupby('region')
    .agg(
        total_students   = ('id_student', 'count'),
        completion_rate  = ('completed', 'mean'),
        avg_score        = ('avg_score', 'mean'),
        withdrawal_rate  = ('withdrew', 'mean'),
        avg_clicks       = ('total_clicks', 'mean')
    )
    .reset_index()
)

geo_df['completion_rate_pct'] = (geo_df['completion_rate'] * 100).round(1)
geo_df['withdrawal_rate_pct'] = (geo_df['withdrawal_rate'] * 100).round(1)
geo_df['avg_score']           = geo_df['avg_score'].round(1)
geo_df['avg_clicks']          = geo_df['avg_clicks'].round(0).astype(int)

geo_df.to_csv(TABLEAU + 'tableau_geo.csv', index=False)
print('Saved tableau_geo.csv')
geo_df.sort_values('total_students', ascending=False)

Saved tableau_geo.csv


,region,total_students,completion_rate,avg_score,withdrawal_rate,avg_clicks,completion_rate_pct,withdrawal_rate_pct
6,Scotland,3446,0.489553,61.2,0.263204,1387,49.0,26.3
0,East Anglian Region,3340,0.489820,57.9,0.301198,1184,49.0,30.1
3,London Region,3216,0.423507,53.5,0.345149,1160,42.4,34.5
8,South Region,3092,0.523933,59.9,0.300129,1316,52.4,30.0
5,North Western Region,2906,0.401927,53.1,0.355816,1123,40.2,35.6
11,West Midlands Region,2582,0.432998,54.8,0.353602,1084,43.3,35.4
9,South West Region,2436,0.497947,58.4,0.311166,1187,49.8,31.1
1,East Midlands Region,2365,0.457082,56.4,0.347569,1105,45.7,34.8
7,South East Region,2111,0.514922,59.5,0.307437,1234,51.5,30.7
10,Wales,2086,0.451582,59.6,0.251198,1066,45.2,25.1


---
## Export 4: Engagement vs Outcome

In [6]:
# Sample to avoid overplotting in Tableau (max 10k rows)
scatter_df = master[[
    'id_student', 'code_module', 'code_presentation',
    'total_clicks', 'active_days', 'avg_score',
    'submission_rate', 'engagement_tier',
    'final_result', 'completed', 'withdrew',
    'gender', 'age_band', 'highest_education', 'region'
]].copy()

# Cap clicks at 99th percentile to avoid extreme outliers distorting chart
p99 = scatter_df['total_clicks'].quantile(0.99)
scatter_df['total_clicks_capped'] = scatter_df['total_clicks'].clip(upper=p99)

if len(scatter_df) > 10000:
    scatter_df = scatter_df.sample(10000, random_state=42)

scatter_df.to_csv(TABLEAU + 'tableau_engagement_vs_result.csv', index=False)
print(f'Saved tableau_engagement_vs_result.csv ({len(scatter_df):,} rows)')

Saved tableau_engagement_vs_result.csv (10,000 rows)


---
## Export 5: Weekly Activity Timeline

In [7]:
sv_with_type = sv.merge(
    vle[['id_site', 'activity_type']],
    on='id_site', how='left'
)

weekly_clicks = (
    sv_with_type
    .groupby(['week', 'activity_type'])['sum_click']
    .sum()
    .reset_index(name='total_clicks')
)

weekly_clicks.to_csv(TABLEAU + 'tableau_weekly_activity.csv', index=False)
print('Saved tableau_weekly_activity.csv')
print(f'Activity types: {weekly_clicks["activity_type"].nunique()}')
weekly_clicks.head(10)

Saved tableau_weekly_activity.csv
Activity types: 20


,week,activity_type,total_clicks
0,0,dataplus,57
1,0,dualpane,427
2,0,externalquiz,4167
3,0,forumng,682793
4,0,glossary,6106
5,0,homepage,399927
6,0,htmlactivity,2705
7,0,oucollaborate,6793
8,0,oucontent,345619
9,0,ouelluminate,2718


In [8]:
# Verify all exports
print('=== Tableau Export Summary ===')
for f in sorted(os.listdir(TABLEAU)):
    if f.endswith('.csv'):
        df = pd.read_csv(TABLEAU + f)
        size = os.path.getsize(TABLEAU + f) / 1024
        print(f'  {f:<45} {df.shape[0]:>6,} rows × {df.shape[1]:>2} cols  ({size:.0f} KB)')

=== Tableau Export Summary ===
  tableau_cohort_retention.csv                     104 rows ×  5 cols  (2 KB)
  tableau_engagement_vs_result.csv              10,000 rows × 16 cols  (1051 KB)
  tableau_funnel.csv                                 5 rows ×  5 cols  (0 KB)
  tableau_geo.csv                                   13 rows ×  8 cols  (1 KB)
  tableau_weekly_activity.csv                      475 rows ×  3 cols  (8 KB)
